# Redshift → Teradata Chunked Load Benchmark

This notebook benchmarks three Python approaches for moving the **same Redshift query result** into Teradata in chunks:

1. **Teradata SQL Driver FastLoad** (`teradatasql`)
2. **Teradata SQL Driver batch INSERT** (`executemany`)
3. **SQLAlchemy + pandas** (`DataFrame.to_sql`)

For every chunk, the benchmark records:

- chunk number
- current chunk row count
- cumulative row count
- DataFrame memory
- Python process RSS memory
- Redshift extraction + DataFrame build time
- Teradata load time
- total chunk time
- average observed extract time
- average observed load time
- elapsed time
- projected total elapsed time
- estimated remaining time (ETA)

## Benchmark design

The source query is re-run once for each method so all three methods process the same data independently.
A `COUNT(*)` over the source query provides the expected row count and expected number of chunks.

**Important:** use a dedicated empty benchmark table for each method. FastLoad is designed for large loads into an empty permanent table and should remain one FastLoad transaction across the chunks.

In [ ]:
# Run once if needed, then restart the kernel if your environment requires it.
%pip install -U pandas numpy psutil sqlalchemy redshift_connector teradatasql teradatasqlalchemy

In [ ]:
from __future__ import annotations

import gc
import math
import os
import time
from dataclasses import dataclass
from datetime import timedelta
from typing import Callable, Iterator

import numpy as np
import pandas as pd
import psutil
import redshift_connector
import teradatasql

from sqlalchemy import create_engine
from sqlalchemy.engine import URL

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

## 1. Configuration

Credentials are read from environment variables rather than hard-coded into the notebook.

Expected variables:

### Redshift
- `RS_HOST`
- `RS_PORT` (optional, defaults to `5439`)
- `RS_DATABASE`
- `RS_USER`
- `RS_PASSWORD`

### Teradata
- `TD_HOST`
- `TD_USER`
- `TD_PASSWORD`
- `TD_DATABASE` / schema

If your enterprise authentication uses LDAP, TDNEGO, Kerberos, IAM, SSO, or a secrets manager, update only the two connection helper functions below.

In [ ]:
# -----------------------------
# Source query
# -----------------------------
SOURCE_SQL = '''
SELECT
    *
FROM your_redshift_schema.your_source_table
WHERE 1 = 1
'''.strip().rstrip(";")

# Optional. If the COUNT(*) wrapper is too expensive, set this to a known row count.
TOTAL_ROWS_OVERRIDE: int | None = None

# -----------------------------
# Chunk / benchmark settings
# -----------------------------
CHUNK_SIZE = 100_000
FASTLOAD_SESSIONS = 8

# Batch INSERT behavior. Committing each chunk is operationally safer for the baseline.
BATCH_COMMIT_EACH_CHUNK = True

# FastLoad error checks add a little benchmark overhead, but prevent silent bad-row loss.
CHECK_FASTLOAD_ERRORS = True

# Garbage collect after every chunk to make memory behavior easier to observe.
GC_EACH_CHUNK = True

# -----------------------------
# Teradata targets
# -----------------------------
TD_SCHEMA = os.getenv("TD_DATABASE", "your_td_database")

TARGET_TABLES = {
    "fastload": "rs_td_bench_fastload",
    "batch_insert": "rs_td_bench_batch",
    "sqlalchemy": "rs_td_bench_sqlalchemy",
}

# Define ONE common physical table structure so the comparison is fair.
# Replace the sample columns below with the actual output columns/types from SOURCE_SQL.
#
# {qualified_table} is replaced automatically for each benchmark method.
TD_CREATE_TABLE_DDL_TEMPLATE = '''
CREATE MULTISET TABLE {qualified_table}
(
    member_id       BIGINT,
    provider_id     BIGINT,
    service_date    DATE,
    metric_value    DECIMAL(18,4),
    description     VARCHAR(500)
)
PRIMARY INDEX (member_id)
'''.strip()

In [ ]:
def require_env(*names: str) -> None:
    missing = [name for name in names if not os.getenv(name)]
    if missing:
        raise EnvironmentError(
            "Missing required environment variables: " + ", ".join(missing)
        )


def get_redshift_connection():
    require_env("RS_HOST", "RS_DATABASE", "RS_USER", "RS_PASSWORD")
    return redshift_connector.connect(
        host=os.environ["RS_HOST"],
        port=int(os.getenv("RS_PORT", "5439")),
        database=os.environ["RS_DATABASE"],
        user=os.environ["RS_USER"],
        password=os.environ["RS_PASSWORD"],
    )


def get_td_connection(*, autocommit: bool = True):
    require_env("TD_HOST", "TD_USER", "TD_PASSWORD")
    con = teradatasql.connect(
        host=os.environ["TD_HOST"],
        user=os.environ["TD_USER"],
        password=os.environ["TD_PASSWORD"],
        database=TD_SCHEMA,
    )
    con.autocommit = autocommit
    return con


def get_td_sqlalchemy_engine():
    require_env("TD_HOST", "TD_USER", "TD_PASSWORD")

    url = URL.create(
        drivername="teradatasql",
        username=os.environ["TD_USER"],
        password=os.environ["TD_PASSWORD"],
        host=os.environ["TD_HOST"],
        database=TD_SCHEMA,
    )
    return create_engine(url, pool_pre_ping=True)


def quote_td_identifier(name: str) -> str:
    return '"' + name.replace('"', '""') + '"'


def qualified_td_table(table_name: str) -> str:
    return f"{quote_td_identifier(TD_SCHEMA)}.{quote_td_identifier(table_name)}"

## 2. Redshift row count and chunk generator

In [ ]:
def get_total_rows(source_sql: str) -> int:
    if TOTAL_ROWS_OVERRIDE is not None:
        return int(TOTAL_ROWS_OVERRIDE)

    count_sql = f"SELECT COUNT(*) FROM ({source_sql}) AS src_count"

    with get_redshift_connection() as con:
        with con.cursor() as cur:
            cur.execute(count_sql)
            return int(cur.fetchone()[0])


def redshift_dataframe_chunks(
    source_sql: str,
    chunk_size: int,
) -> Iterator[tuple[pd.DataFrame, float]]:
    """
    Yield (DataFrame, extract_seconds).

    extract_seconds includes:
      - Redshift fetchmany()
      - construction of the pandas DataFrame
    """
    with get_redshift_connection() as con:
        with con.cursor() as cur:
            cur.execute(source_sql)
            columns = [desc[0] for desc in cur.description]

            while True:
                t0 = time.perf_counter()
                rows = cur.fetchmany(chunk_size)

                if not rows:
                    break

                df = pd.DataFrame.from_records(rows, columns=columns)
                extract_seconds = time.perf_counter() - t0

                yield df, extract_seconds

## 3. Target-table utilities

In [ ]:
def reset_target_table(table_name: str) -> None:
    qtable = qualified_td_table(table_name)
    ddl = TD_CREATE_TABLE_DDL_TEMPLATE.format(qualified_table=qtable)

    with get_td_connection(autocommit=True) as con:
        with con.cursor() as cur:
            cur.execute(f"DROP TABLE {qtable}", ignoreErrors=[3807])
            cur.execute(ddl)


def td_table_row_count(table_name: str) -> int:
    qtable = qualified_td_table(table_name)

    with get_td_connection(autocommit=True) as con:
        with con.cursor() as cur:
            cur.execute(f"SELECT COUNT(*) FROM {qtable}")
            return int(cur.fetchone()[0])


def build_parameterized_insert(table_name: str, columns: list[str]) -> str:
    qtable = qualified_td_table(table_name)
    qcols = ", ".join(quote_td_identifier(c) for c in columns)
    markers = ", ".join("?" for _ in columns)
    return f"INSERT INTO {qtable} ({qcols}) VALUES ({markers})"


def dataframe_to_bind_rows(df: pd.DataFrame) -> list[tuple]:
    """Convert pandas/NumPy values into DB-API-friendly Python values."""
    clean = df.astype(object).where(pd.notna(df), None)

    def py_value(v):
        if isinstance(v, np.generic):
            return v.item()
        if isinstance(v, pd.Timestamp):
            return v.to_pydatetime()
        return v

    return [
        tuple(py_value(v) for v in row)
        for row in clean.itertuples(index=False, name=None)
    ]

## 4. Progress, memory, ETA, and benchmark result helpers

In [ ]:
PROCESS = psutil.Process(os.getpid())


def fmt_seconds(seconds: float | None) -> str:
    if seconds is None or not np.isfinite(seconds):
        return "n/a"
    return str(timedelta(seconds=int(round(seconds))))


def df_memory_mb(df: pd.DataFrame) -> float:
    return float(df.memory_usage(index=True, deep=True).sum() / (1024 ** 2))


def process_rss_mb() -> float:
    return float(PROCESS.memory_info().rss / (1024 ** 2))


@dataclass
class BenchmarkContext:
    method: str
    total_rows: int
    chunk_size: int
    expected_chunks: int
    started_at: float
    metrics: list[dict]


def record_chunk(
    ctx: BenchmarkContext,
    *,
    chunk_no: int,
    df: pd.DataFrame,
    cumulative_rows: int,
    extract_s: float,
    load_s: float,
) -> None:
    chunk_s = extract_s + load_s

    avg_extract_s = np.mean([m["extract_s"] for m in ctx.metrics] + [extract_s])
    avg_load_s = np.mean([m["load_s"] for m in ctx.metrics] + [load_s])
    avg_chunk_s = avg_extract_s + avg_load_s

    elapsed_s = time.perf_counter() - ctx.started_at
    remaining_chunks = max(ctx.expected_chunks - chunk_no, 0)

    projected_total_s = avg_chunk_s * ctx.expected_chunks
    eta_s = avg_chunk_s * remaining_chunks

    row = {
        "method": ctx.method,
        "chunk": chunk_no,
        "chunk_rows": len(df),
        "cumulative_rows": cumulative_rows,
        "total_rows_expected": ctx.total_rows,
        "progress_pct": 100 * cumulative_rows / ctx.total_rows if ctx.total_rows else np.nan,
        "df_memory_mb": df_memory_mb(df),
        "process_rss_mb": process_rss_mb(),
        "extract_s": extract_s,
        "load_s": load_s,
        "chunk_total_s": chunk_s,
        "avg_extract_s": avg_extract_s,
        "avg_load_s": avg_load_s,
        "elapsed_s": elapsed_s,
        "projected_total_s": projected_total_s,
        "eta_s": eta_s,
    }
    ctx.metrics.append(row)

    print(
        f"[{ctx.method}] "
        f"chunk {chunk_no:,}/{ctx.expected_chunks:,} | "
        f"rows {len(df):,} | "
        f"cumulative {cumulative_rows:,}/{ctx.total_rows:,} "
        f"({row['progress_pct']:.1f}%) | "
        f"DF {row['df_memory_mb']:.1f} MB | "
        f"RSS {row['process_rss_mb']:.1f} MB | "
        f"extract {extract_s:.2f}s | "
        f"load {load_s:.2f}s | "
        f"elapsed {fmt_seconds(elapsed_s)} | "
        f"projected {fmt_seconds(projected_total_s)} | "
        f"ETA {fmt_seconds(eta_s)}"
    )


def finalize_benchmark(
    ctx: BenchmarkContext,
    target_table: str,
) -> tuple[pd.DataFrame, dict]:
    detail = pd.DataFrame(ctx.metrics)
    elapsed = time.perf_counter() - ctx.started_at
    td_rows = td_table_row_count(target_table)

    summary = {
        "method": ctx.method,
        "source_rows_expected": ctx.total_rows,
        "target_rows": td_rows,
        "row_count_match": td_rows == ctx.total_rows,
        "chunks": len(detail),
        "elapsed_s": elapsed,
        "rows_per_second": td_rows / elapsed if elapsed else np.nan,
        "avg_extract_s": detail["extract_s"].mean() if not detail.empty else np.nan,
        "avg_load_s": detail["load_s"].mean() if not detail.empty else np.nan,
        "peak_df_memory_mb": detail["df_memory_mb"].max() if not detail.empty else np.nan,
        "peak_process_rss_mb": detail["process_rss_mb"].max() if not detail.empty else np.nan,
    }
    return detail, summary

## 5. Method 1. Teradata SQL Driver FastLoad

This uses the Teradata driver's FastLoad protocol directly.

- target table starts empty
- auto-commit is disabled
- each Redshift DataFrame becomes another batch in the **same FastLoad**
- commit occurs after the final chunk
- optional acquisition-phase error checking occurs after every chunk

Use a sufficiently large chunk size to amortize FastLoad setup overhead.

In [ ]:
def get_fastload_messages(cur, fastload_insert_sql: str, kind: str) -> list:
    if kind not in {"errors", "warnings"}:
        raise ValueError("kind must be 'errors' or 'warnings'")

    request = (
        f"{{fn teradata_nativesql}}"
        f"{{fn teradata_get_{kind}}}"
        f"{fastload_insert_sql}"
    )
    cur.execute(request)
    return cur.fetchall()


def benchmark_fastload(
    source_sql: str,
    target_table: str,
    *,
    chunk_size: int = CHUNK_SIZE,
    sessions: int = FASTLOAD_SESSIONS,
) -> tuple[pd.DataFrame, dict]:

    reset_target_table(target_table)

    total_rows = get_total_rows(source_sql)
    expected_chunks = math.ceil(total_rows / chunk_size) if total_rows else 0

    ctx = BenchmarkContext(
        method="FastLoad",
        total_rows=total_rows,
        chunk_size=chunk_size,
        expected_chunks=expected_chunks,
        started_at=time.perf_counter(),
        metrics=[],
    )

    cumulative_rows = 0
    insert_sql = None

    with get_td_connection(autocommit=False) as td_con:
        with td_con.cursor() as td_cur:
            try:
                for chunk_no, (df, extract_s) in enumerate(
                    redshift_dataframe_chunks(source_sql, chunk_size),
                    start=1,
                ):
                    if insert_sql is None:
                        base_insert = build_parameterized_insert(
                            target_table,
                            df.columns.tolist(),
                        )
                        insert_sql = (
                            f"{{fn teradata_require_fastload}}"
                            f"{{fn teradata_sessions({sessions})}}"
                            f"{base_insert}"
                        )

                    bind_rows = dataframe_to_bind_rows(df)

                    load_t0 = time.perf_counter()
                    td_cur.executemany(insert_sql, bind_rows)

                    if CHECK_FASTLOAD_ERRORS:
                        errors = get_fastload_messages(td_cur, insert_sql, "errors")
                        if errors:
                            raise RuntimeError(
                                f"FastLoad acquisition errors in chunk {chunk_no}: {errors[:10]}"
                            )

                    load_s = time.perf_counter() - load_t0

                    cumulative_rows += len(df)
                    record_chunk(
                        ctx,
                        chunk_no=chunk_no,
                        df=df,
                        cumulative_rows=cumulative_rows,
                        extract_s=extract_s,
                        load_s=load_s,
                    )

                    del bind_rows, df
                    if GC_EACH_CHUNK:
                        gc.collect()

                td_con.commit()

                if CHECK_FASTLOAD_ERRORS and insert_sql is not None:
                    apply_errors = get_fastload_messages(td_cur, insert_sql, "errors")
                    if apply_errors:
                        raise RuntimeError(
                            f"FastLoad apply-phase errors: {apply_errors[:10]}"
                        )

            except Exception:
                td_con.rollback()
                raise

    return finalize_benchmark(ctx, target_table)

## 6. Method 2. Teradata SQL Driver batch INSERT

In [ ]:
def benchmark_batch_insert(
    source_sql: str,
    target_table: str,
    *,
    chunk_size: int = CHUNK_SIZE,
    commit_each_chunk: bool = BATCH_COMMIT_EACH_CHUNK,
) -> tuple[pd.DataFrame, dict]:

    reset_target_table(target_table)

    total_rows = get_total_rows(source_sql)
    expected_chunks = math.ceil(total_rows / chunk_size) if total_rows else 0

    ctx = BenchmarkContext(
        method="Batch INSERT",
        total_rows=total_rows,
        chunk_size=chunk_size,
        expected_chunks=expected_chunks,
        started_at=time.perf_counter(),
        metrics=[],
    )

    cumulative_rows = 0
    insert_sql = None

    with get_td_connection(autocommit=False) as td_con:
        with td_con.cursor() as td_cur:
            try:
                for chunk_no, (df, extract_s) in enumerate(
                    redshift_dataframe_chunks(source_sql, chunk_size),
                    start=1,
                ):
                    if insert_sql is None:
                        insert_sql = build_parameterized_insert(
                            target_table,
                            df.columns.tolist(),
                        )

                    bind_rows = dataframe_to_bind_rows(df)

                    load_t0 = time.perf_counter()
                    td_cur.executemany(insert_sql, bind_rows)

                    if commit_each_chunk:
                        td_con.commit()

                    load_s = time.perf_counter() - load_t0

                    cumulative_rows += len(df)
                    record_chunk(
                        ctx,
                        chunk_no=chunk_no,
                        df=df,
                        cumulative_rows=cumulative_rows,
                        extract_s=extract_s,
                        load_s=load_s,
                    )

                    del bind_rows, df
                    if GC_EACH_CHUNK:
                        gc.collect()

                if not commit_each_chunk:
                    td_con.commit()

            except Exception:
                td_con.rollback()
                raise

    return finalize_benchmark(ctx, target_table)

## 7. Method 3. SQLAlchemy + pandas `to_sql`

In [ ]:
def benchmark_sqlalchemy(
    source_sql: str,
    target_table: str,
    *,
    chunk_size: int = CHUNK_SIZE,
) -> tuple[pd.DataFrame, dict]:

    reset_target_table(target_table)

    total_rows = get_total_rows(source_sql)
    expected_chunks = math.ceil(total_rows / chunk_size) if total_rows else 0

    ctx = BenchmarkContext(
        method="SQLAlchemy + pandas.to_sql",
        total_rows=total_rows,
        chunk_size=chunk_size,
        expected_chunks=expected_chunks,
        started_at=time.perf_counter(),
        metrics=[],
    )

    cumulative_rows = 0
    engine = get_td_sqlalchemy_engine()

    try:
        for chunk_no, (df, extract_s) in enumerate(
            redshift_dataframe_chunks(source_sql, chunk_size),
            start=1,
        ):
            load_t0 = time.perf_counter()

            df.to_sql(
                name=target_table,
                schema=TD_SCHEMA,
                con=engine,
                if_exists="append",
                index=False,
                method=None,
            )

            load_s = time.perf_counter() - load_t0

            cumulative_rows += len(df)
            record_chunk(
                ctx,
                chunk_no=chunk_no,
                df=df,
                cumulative_rows=cumulative_rows,
                extract_s=extract_s,
                load_s=load_s,
            )

            del df
            if GC_EACH_CHUNK:
                gc.collect()

    finally:
        engine.dispose()

    return finalize_benchmark(ctx, target_table)

## 8. Sanity check before running the full benchmark

Start with a smaller source query or a `LIMIT` while validating:

1. Redshift connectivity
2. target DDL and column order
3. datatype compatibility
4. duplicate-row behavior
5. Teradata permissions
6. FastLoad prerequisites

Then remove the test `LIMIT` and run the real benchmark.

In [ ]:
TOTAL_ROWS = get_total_rows(SOURCE_SQL)
EXPECTED_CHUNKS = math.ceil(TOTAL_ROWS / CHUNK_SIZE) if TOTAL_ROWS else 0

print(f"Expected rows   : {TOTAL_ROWS:,}")
print(f"Chunk size      : {CHUNK_SIZE:,}")
print(f"Expected chunks : {EXPECTED_CHUNKS:,}")

## 9. Run the three benchmarks

In [ ]:
fastload_detail, fastload_summary = benchmark_fastload(
    SOURCE_SQL,
    TARGET_TABLES["fastload"],
    chunk_size=CHUNK_SIZE,
    sessions=FASTLOAD_SESSIONS,
)

In [ ]:
batch_detail, batch_summary = benchmark_batch_insert(
    SOURCE_SQL,
    TARGET_TABLES["batch_insert"],
    chunk_size=CHUNK_SIZE,
    commit_each_chunk=BATCH_COMMIT_EACH_CHUNK,
)

In [ ]:
sqlalchemy_detail, sqlalchemy_summary = benchmark_sqlalchemy(
    SOURCE_SQL,
    TARGET_TABLES["sqlalchemy"],
    chunk_size=CHUNK_SIZE,
)

## 10. Benchmark comparison

In [ ]:
benchmark_summary = pd.DataFrame([
    fastload_summary,
    batch_summary,
    sqlalchemy_summary,
]).sort_values("elapsed_s")

benchmark_summary["elapsed"] = benchmark_summary["elapsed_s"].map(fmt_seconds)

display(
    benchmark_summary[
        [
            "method",
            "source_rows_expected",
            "target_rows",
            "row_count_match",
            "chunks",
            "elapsed",
            "rows_per_second",
            "avg_extract_s",
            "avg_load_s",
            "peak_df_memory_mb",
            "peak_process_rss_mb",
        ]
    ].reset_index(drop=True)
)

In [ ]:
benchmark_detail = pd.concat(
    [fastload_detail, batch_detail, sqlalchemy_detail],
    ignore_index=True,
)

display(
    benchmark_detail[
        [
            "method",
            "chunk",
            "chunk_rows",
            "cumulative_rows",
            "progress_pct",
            "df_memory_mb",
            "process_rss_mb",
            "extract_s",
            "load_s",
            "chunk_total_s",
            "elapsed_s",
            "projected_total_s",
            "eta_s",
        ]
    ]
)

## 11. Optional charts

In [ ]:
import matplotlib.pyplot as plt

summary_plot = benchmark_summary.sort_values("elapsed_s")

plt.figure(figsize=(9, 5))
plt.bar(summary_plot["method"], summary_plot["rows_per_second"])
plt.ylabel("Rows / second")
plt.title("Redshift → Teradata Throughput")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
for method, g in benchmark_detail.groupby("method"):
    plt.figure(figsize=(9, 5))
    plt.plot(g["chunk"], g["process_rss_mb"], marker="o")
    plt.xlabel("Chunk")
    plt.ylabel("Process RSS (MB)")
    plt.title(f"Memory Profile: {method}")
    plt.tight_layout()
    plt.show()

## 12. How to interpret the result

### FastLoad
Best candidate for very large transfers when the target/load pattern satisfies FastLoad restrictions.
Pay attention to first-chunk overhead because FastLoad opens additional transfer sessions.

### Batch INSERT
Useful low-level baseline. Simpler and more flexible than FastLoad, but usually slower at large scale.

### SQLAlchemy + pandas
Best developer ergonomics and maintainability baseline. It may trade throughput for abstraction.

### What to tune next

Repeat with several chunk sizes, for example:

- `25_000`
- `50_000`
- `100_000`
- `250_000`

Choose the point where throughput stops improving materially or process RSS becomes uncomfortable.

Also compare:

- target-table PI and skew
- network throughput
- Redshift query execution vs fetch time
- datatype-conversion cost
- transaction / rollback requirements
- duplicate-row behavior
- workload-management limits

## 13. Optional automated chunk-size testing

In [ ]:
def benchmark_chunk_sizes(
    method_fn: Callable,
    source_sql: str,
    target_table: str,
    chunk_sizes: list[int],
    **kwargs,
) -> pd.DataFrame:
    results = []

    for size in chunk_sizes:
        print("\n" + "=" * 90)
        print(f"Testing chunk size: {size:,}")
        print("=" * 90)

        _, summary = method_fn(
            source_sql,
            target_table,
            chunk_size=size,
            **kwargs,
        )
        summary["chunk_size"] = size
        results.append(summary)

    return (
        pd.DataFrame(results)
        .sort_values("rows_per_second", ascending=False)
        .reset_index(drop=True)
    )


# Example:
# fastload_chunk_test = benchmark_chunk_sizes(
#     benchmark_fastload,
#     SOURCE_SQL,
#     TARGET_TABLES["fastload"],
#     chunk_sizes=[25_000, 50_000, 100_000, 250_000],
#     sessions=FASTLOAD_SESSIONS,
# )
# display(fastload_chunk_test)